# EDA — Detección de Outliers a Nivel de Píxel por Municipio
## CHIRPS — Precipitación mensual | Enero 2025

**Objetivo:** Explorar la distribución de valores de precipitación a nivel de píxel dentro de municipios seleccionados aleatoriamente, identificar la presencia de valores atípicos y evaluar el impacto de su eliminación sobre las estadísticas agregadas municipales.

**Referencia metodológica:** Se utiliza el método IQR (rango intercuartílico) para la detección de outliers, apropiado para datos de precipitación dado que no siguen una distribución normal (Wilks, 2011).

## 0. Configuración de rutas y parámetros

In [1]:
from pathlib import Path

# Ajusta estas rutas según tu entorno local
PROJECT_ROOT  = Path("C:/Users/laura/OneDrive/TESIS/ETL_LauraChacon/ETL_code")
TIF_PATH      = PROJECT_ROOT / "data/raw/chirps_raster_raw/chirps-v2.0.2025.01.tif"
CONFIG_PATH   = PROJECT_ROOT / "config/config.yaml"
OUTPUT_DIR    = PROJECT_ROOT / "data/processed/eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_MUNICIPIOS  = 12
RANDOM_SEED   = 42
IQR_FACTOR    = 1.5
NODATA_VALUE  = -9999

print("Rutas configuradas.")
print(f"  TIF   : {TIF_PATH}")
print(f"  Config: {CONFIG_PATH}")
print(f"  Output: {OUTPUT_DIR}")

Rutas configuradas.
  TIF   : C:\Users\laura\OneDrive\TESIS\ETL_LauraChacon\ETL_code\data\raw\chirps_raster_raw\chirps-v2.0.2025.01.tif
  Config: C:\Users\laura\OneDrive\TESIS\ETL_LauraChacon\ETL_code\config\config.yaml
  Output: C:\Users\laura\OneDrive\TESIS\ETL_LauraChacon\ETL_code\data\processed\eda


## 1. Importaciones

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rasterio_mask
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from src.utils.config_loader import load_config
from src.etl.transform.transform_chirps import load_municipalities_gdf

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Importaciones completadas.")

## 2. Carga del raster CHIRPS y del GeoDataFrame municipal

In [ ]:
config = load_config(CONFIG_PATH)

gdf_municipios = load_municipalities_gdf(config, project_root=PROJECT_ROOT)
print(f"Municipios cargados: {len(gdf_municipios):,}")
print(f"CRS municipios     : {gdf_municipios.crs}")

if not TIF_PATH.exists():
    raise FileNotFoundError(f"No se encontro el TIF: {TIF_PATH}")

with rasterio.open(TIF_PATH) as src:
    raster_crs    = src.crs
    nodata_raster = src.nodata if src.nodata is not None else NODATA_VALUE
    bounds        = src.bounds
    resolution    = src.res
    print(f"
Raster CHIRPS:")
    print(f"  CRS       : {raster_crs}")
    print(f"  Resolucion: {resolution[0]:.4f} x {resolution[1]:.4f} grad (~{resolution[0]*111:.1f} km)")
    print(f"  Nodata    : {nodata_raster}")
    print(f"  Bounds    : {bounds}")

## 3. Selección aleatoria de municipios

Se seleccionan municipios de distintas regiones geográficas para garantizar diversidad climática en el análisis.

In [ ]:
np.random.seed(RANDOM_SEED)

if gdf_municipios.crs != raster_crs:
    gdf_municipios = gdf_municipios.to_crs(raster_crs)
    print(f"Municipios reproyectados a: {raster_crs}")

muestra = gdf_municipios.sample(n=N_MUNICIPIOS, random_state=RANDOM_SEED).copy()
muestra = muestra.reset_index(drop=True)

print(f"Municipios seleccionados ({N_MUNICIPIOS}):")
cols_mostrar = ["muni_code"] + [c for c in ["nombre_mpio", "nombre_dpto", "dpto"] if c in muestra.columns]
print(muestra[cols_mostrar].to_string(index=False))

## 4. Extracción de píxeles por municipio

Para cada municipio se extraen los valores de precipitación a nivel de píxel usando  con , igual que en el pipeline de transformación.

In [ ]:
registros = []

with rasterio.open(TIF_PATH) as src:
    for _, muni in muestra.iterrows():
        muni_code = muni["muni_code"]
        geom      = [muni.geometry.__geo_interface__]

        try:
            data, _ = rasterio_mask(
                src, geom,
                crop=True, filled=True,
                nodata=nodata_raster,
                all_touched=True
            )
            band  = data[0]
            flat  = band.astype("float32").ravel()
            valid = flat[flat != nodata_raster]
            valid = valid[~np.isnan(valid)]

            if valid.size == 0:
                print(f"  [AVISO] muni_code={muni_code}: sin pixeles validos.")
                continue

            Q1    = np.percentile(valid, 25)
            Q3    = np.percentile(valid, 75)
            IQR   = Q3 - Q1
            lower = Q1 - IQR_FACTOR * IQR
            upper = Q3 + IQR_FACTOR * IQR
            clean = valid[(valid >= lower) & (valid <= upper)]
            n_out = valid.size - clean.size

            registros.append({
                "muni_code"      : muni_code,
                "n_pixels_total" : int(valid.size),
                "n_outliers"     : int(n_out),
                "pct_outliers"   : round(n_out / valid.size * 100, 2),
                "mean_raw"       : float(valid.mean()),
                "min_raw"        : float(valid.min()),
                "max_raw"        : float(valid.max()),
                "std_raw"        : float(valid.std(ddof=0)),
                "mean_clean"     : float(clean.mean()) if clean.size > 0 else float("nan"),
                "min_clean"      : float(clean.min())  if clean.size > 0 else float("nan"),
                "max_clean"      : float(clean.max())  if clean.size > 0 else float("nan"),
                "std_clean"      : float(clean.std(ddof=0)) if clean.size > 0 else float("nan"),
                "iqr_lower"      : float(lower),
                "iqr_upper"      : float(upper),
                "_pixels_raw"    : valid,
                "_pixels_clean"  : clean,
            })

        except Exception as e:
            print(f"  [ERROR] muni_code={muni_code}: {e}")

print(f"Municipios procesados: {len(registros)}/{N_MUNICIPIOS}")

## 5. Tabla resumen: estadísticas con y sin outliers

In [ ]:
df_resumen = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in registros
])

df_resumen["delta_mean_pct"] = (
    (df_resumen["mean_clean"] - df_resumen["mean_raw"])
    / df_resumen["mean_raw"].replace(0, float("nan")) * 100
).round(2)

display_cols = [
    "muni_code", "n_pixels_total", "n_outliers", "pct_outliers",
    "mean_raw", "mean_clean", "delta_mean_pct",
    "std_raw", "std_clean"
]

print("Tabla resumen (precipitacion en mm):")
print(df_resumen[display_cols].to_string(index=False))

csv_path = OUTPUT_DIR / "eda_outliers_enero2025.csv"
df_resumen[[c for c in df_resumen.columns if not c.startswith("_")]].to_csv(csv_path, index=False)
print(f"
CSV guardado en: {csv_path}")

## 6. Visualización: distribución de píxeles por municipio

Para cada municipio se muestra el histograma con los límites IQR marcados. Los píxeles en rojo son outliers.

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(registros) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, r in enumerate(registros):
    ax       = axes[i]
    raw      = r["_pixels_raw"]
    mask_out = (raw < r["iqr_lower"]) | (raw > r["iqr_upper"])
    inliers  = raw[~mask_out]
    outliers = raw[mask_out]

    ax.hist(inliers, bins=30, color="#4C72B0", alpha=0.8, label="Inliers")
    if outliers.size > 0:
        ax.hist(outliers, bins=15, color="#DD4444", alpha=0.7, label=f"Outliers ({outliers.size})")

    ax.axvline(r["iqr_lower"], color="orange", linestyle="--", linewidth=1.2,
               label=f"Q1-1.5xIQR ({r['iqr_lower']:.1f})")
    ax.axvline(r["iqr_upper"], color="green",  linestyle="--", linewidth=1.2,
               label=f"Q3+1.5xIQR ({r['iqr_upper']:.1f})")
    ax.axvline(r["mean_raw"],   color="#4C72B0", linestyle=":", linewidth=1.5)
    ax.axvline(r["mean_clean"], color="#22AA66", linestyle=":", linewidth=1.5)

    ax.set_title(f"Muni: {r['muni_code']}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Precipitacion (mm)", fontsize=8)
    ax.set_ylabel("Frecuencia", fontsize=8)
    ax.legend(fontsize=7, loc="upper right")
    ax.tick_params(labelsize=8)

for j in range(len(registros), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Distribucion de pixeles CHIRPS por municipio - Enero 2025
"
    "(linea punteada azul = media con outliers | verde = media sin outliers)",
    fontsize=11, fontweight="bold", y=1.01
)
plt.tight_layout()
plot_path = OUTPUT_DIR / "distribucion_pixeles_municipios.png"
plt.savefig(plot_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Figura guardada: {plot_path}")

## 7. Impacto de la eliminación de outliers sobre la media municipal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x     = np.arange(len(df_resumen))
width = 0.35

ax1 = axes[0]
ax1.bar(x - width/2, df_resumen["mean_raw"],   width, label="Con outliers",  color="#4C72B0", alpha=0.85)
ax1.bar(x + width/2, df_resumen["mean_clean"], width, label="Sin outliers",  color="#22AA66", alpha=0.85)
ax1.set_xlabel("Municipio", fontsize=9)
ax1.set_ylabel("Precipitacion media (mm)", fontsize=9)
ax1.set_title("Media municipal: con vs. sin outliers", fontsize=10, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(df_resumen["muni_code"], rotation=45, ha="right", fontsize=8)
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.4)

ax2    = axes[1]
deltas = df_resumen["delta_mean_pct"].fillna(0)
colors = ["#DD4444" if d < 0 else "#22AA66" for d in deltas]
ax2.bar(x, deltas, color=colors, alpha=0.85)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Municipio", fontsize=9)
ax2.set_ylabel("Delta media (%)", fontsize=9)
ax2.set_title("Diferencia porcentual en la media", fontsize=10, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(df_resumen["muni_code"], rotation=45, ha="right", fontsize=8)
ax2.grid(axis="y", alpha=0.4)
legend_elements = [
    Patch(facecolor="#DD4444", alpha=0.85, label="Outliers sobreestimaban la media"),
    Patch(facecolor="#22AA66", alpha=0.85, label="Outliers subestimaban la media"),
]
ax2.legend(handles=legend_elements, fontsize=8)

plt.tight_layout()
plot2_path = OUTPUT_DIR / "impacto_outliers_media.png"
plt.savefig(plot2_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Figura guardada: {plot2_path}")

## 8. Boxplots por municipio — identificación visual de outliers

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

data_box   = [r["_pixels_raw"] for r in registros]
labels_box = [r["muni_code"]   for r in registros]

ax.boxplot(
    data_box, labels=labels_box,
    patch_artist=True, notch=False, showfliers=True,
    flierprops=dict(marker="o", color="#DD4444", alpha=0.5, markersize=3),
    medianprops=dict(color="#22AA66", linewidth=2),
    boxprops=dict(facecolor="#4C72B0", alpha=0.5),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
)
ax.set_xlabel("Municipio (muni_code)", fontsize=9)
ax.set_ylabel("Precipitacion por pixel (mm)", fontsize=9)
ax.set_title(
    "Boxplot de valores de precipitacion por pixel - Enero 2025
"
    "(puntos rojos = outliers por regla de Tukey IQR x1.5)",
    fontsize=10, fontweight="bold"
)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.grid(axis="y", alpha=0.4)

plt.tight_layout()
plot3_path = OUTPUT_DIR / "boxplots_municipios.png"
plt.savefig(plot3_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Figura guardada: {plot3_path}")

## 9. Conclusiones del EDA

In [ ]:
print("=" * 60)
print("RESUMEN DEL ANALISIS DE OUTLIERS - CHIRPS Enero 2025")
print("=" * 60)

total_pixeles  = df_resumen["n_pixels_total"].sum()
total_outliers = df_resumen["n_outliers"].sum()
pct_global     = total_outliers / total_pixeles * 100
max_impacto    = df_resumen["delta_mean_pct"].abs().max()
muni_max       = df_resumen.loc[df_resumen["delta_mean_pct"].abs().idxmax(), "muni_code"]
sin_outliers   = (df_resumen["n_outliers"] == 0).sum()

pct_str     = f"{pct_global:.2f}"
impacto_str = f"{max_impacto:.2f}"

print(f"
Municipios analizados   : {len(df_resumen)}")
print(f"Total pixeles validos   : {total_pixeles:,}")
print(f"Total outliers IQR      : {total_outliers:,} ({pct_str}% del total)")
print(f"Municipios sin outliers : {sin_outliers} de {len(df_resumen)}")
print(f"
Municipio con mayor impacto: {muni_max} ({impacto_str}%)")
print()
print("Municipios con >5% de outliers:")
alto = df_resumen[df_resumen["pct_outliers"] > 5][
    ["muni_code","n_pixels_total","n_outliers","pct_outliers","delta_mean_pct"]
]
if alto.empty:
    print("  Ninguno — proporcion baja en todos los municipios.")
else:
    print(alto.to_string(index=False))

print()
print("INTERPRETACION:")
print(f"1. El {pct_str}% de pixeles fue identificado como outlier (IQR x{IQR_FACTOR}).")
print(f"2. La mayor diferencia en la media fue {impacto_str}% (municipio {muni_max}).")
print("3. Diferencias <5% indican impacto limitado sobre el repositorio.")
print("4. Si se identifican municipios con alto porcentaje de outliers,")
print("   incorporar filtro IQR antes del calculo de estadisticas zonales")
print("   es recomendable como mejora del pipeline.")